# 6 WorkFlow  Gerencial, futuro=JULIO

### 6.1 Objetivo

Presentar un workflow/pipeline completo al que los estudiantes deberán enriquecer

#### 6.2  Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Drive already mounted at /content/.drive; to attempt to forcibly remount, call drive.mount("/content/.drive", force_remount=True).


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab
*   Bajar el **dataset_historico** al Google Drive y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/dm"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dm"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/itba2026-7c9a/dm/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}


# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"
descargar  "gerencial_competencia_2026.csv.gz"

cp: cannot stat '/content/buckets/b1/kaggle/kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
--2026-09-14 12:37:18--  https://storage.googleapis.com/open-courses/itba2026-7c9a/dm/dataset_pequeno.csv
Resolving storage.googleapis.com (storage.googleapis.com)... 192.178.210.207, 74.125.132.207, 192.178.129.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|192.178.210.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 167807380 (160M) [text/csv]
Saving to: ‘/content/buckets/b1/datasets/dataset_pequeno.csv’

/content/buckets/b1 100%[===================>] 160.03M   208MB/s    in 0.8s    

2026-09-14 12:37:19 (208 MB/s) - ‘/content/buckets/b1/datasets/dataset_pequeno.csv’ saved [167807380/167807380]

--2026-09-14 12:37:25--  https://storage.googleapis.com/open-courses/itba2026-7c9a/dm/gerencial_competencia_2026.csv.gz
Resolving storage.googleapis.com (storage.googleapis.com)... 1

## 6.3  Workflow

## Inicializacion

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Sep 14 12:37:38 PM 2026"

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671284,35.9,1473300,78.7,1473300,78.7
Vcells,1242656,9.5,8388608,64.0,1978689,15.1


In [ ]:
require("data.table")

if( !require("R.utils")) install.packages("R.utils")
require("R.utils")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: R.utils

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘R.utils’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘R.oo’, ‘R.methodsS3’


Loading required package: R.utils

Loading required package: R.oo

Loading required package: R.methodsS3

R.methodsS3 v1.8.2 (2022-06-13 22:00:14 UTC) successfully loaded. See ?R.methodsS3 for help.

R.oo v1.27.1 (2025-05-02 21:00:05 UTC) successfully loaded. See ?R.oo for help.


Attaching package: ‘R.oo’


The following object is masked from ‘package:R.methodsS3’:

    throw


The following objects are masked from ‘package:methods’:

    getClasses, getMethods


The following objects are masked from ‘package:base’:

    attach, detach,

#### Parametros
Si es gerente, no cambie nada
<br>Si es Analista, cambie el nombre del dataset

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 130031

PARAM$experimento <- 6311
PARAM$dataset <- "gerencial_competencia_2026.csv.gz"


#### Carpeta del Experimento

In [ ]:
# carpeta de trabajo

setwd("/content/buckets/b1/exp")
experimento_folder <- paste0("WF", PARAM$experimento)
dir.create(experimento_folder, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento_folder ))

### 6.3.1   Preprocesamiento del dataset

#### 6.3.1.1  DT incorporar dataset

In [ ]:
# lectura del dataset
dataset <- fread(paste0("/content/datasets/", PARAM$dataset))

#### 6.3.1.2  CA  Catastrophe Analysis
Se intentan reparar las variables que para un mes están con todos los valores en cero.

El método que se utiliza es **Machine Learning** se asigna NA also valores, si ha leido bien, es la "anti imputación de valores faltantes"
<br> Usted podrá aplicar aquí otros métodos

In [ ]:
dataset[ foto_mes==202006, internet:=NA]
dataset[ foto_mes==202006, mrentabilidad:=NA]
dataset[ foto_mes==202006, mrentabilidad_annual:=NA]
dataset[ foto_mes==202006, mcomisiones:=NA]
dataset[ foto_mes==202006, mactivos_margen:=NA]
dataset[ foto_mes==202006, mpasivos_margen:=NA]
dataset[ foto_mes==202006, mcuentas_saldo:=NA]
dataset[ foto_mes==202006, ctarjeta_visa_transacciones:=NA]
dataset[ foto_mes==202006, mtarjeta_visa_consumo:=NA]
dataset[ foto_mes==202006, mtarjeta_master_consumo:=NA]
dataset[ foto_mes==202006, ccallcenter_transacciones:=NA]
dataset[ foto_mes==202006, chomebanking_transacciones:=NA]
dataset[ foto_mes==202006, chomebanking_transacciones:=NA]

#### 6.3.1.3  DR  Data Drifting
Se intenta corregir el data drifting, quizas ajustando por IPC ...
<br>Esta parte podrá ser abordada por todos los Analistas y también la Gerenciapero se decide pedagogicamente no incluirla en esta primer version para reducir la carga cognitiva

In [ ]:
# sin codigo en esta primera version del workflow

#### 6.3.1.3  FE_intra_manual Feature Engineering intra-mes

Agrego campos nuevos dentro del mismo mes, SIN considerar la historia.

In [ ]:
# esta funcion atributos presentes existe debido a que las modalidades poseen datasets con distinta cantidad de campos
atributos_presentes <- function( patributos )
{
  atributos <- unique( patributos )
  comun <- intersect( atributos, colnames(dataset) )

  return(  length( atributos ) == length( comun ) )
}

# division segura: evita Inf/NaN cuando el denominador es 0 (queda NA)
divseg <- function( numerador, denominador )
{
  denominador[ denominador == 0 ] <- NA
  return( numerador / denominador )
}

# el mes 1,2, ..12
if( atributos_presentes( c("foto_mes") ))
  dataset[, kmes := foto_mes %% 100]

# variable extraida de una tesis de maestria de Irlanda
if( atributos_presentes( c("mpayroll", "cliente_edad") ))
  dataset[, mpayroll_sobre_edad := mpayroll / cliente_edad]


# ---------------------------------------------------------------------
# FE intra-mes basado en Feature Importance del baseline (10 semillas)
#
# Se toman las variables mas importantes del baseline (Gain/Frecuencia
# promedio en el TOP 10 de las 10 semillas corridas), EXCLUYENDO
# ctrx_quarter_lag1 porque no es intra-mes (proviene del FE historico,
# que en esta etapa no esta permitido usar).
#
# Quedan 10 variables intra-mes:
#   ctrx_quarter, mcaja_ahorro, cpayroll_trx, mcuentas_saldo,
#   mcuenta_corriente, mprestamos_personales, mtarjeta_visa_consumo,
#   Visa_mpagominimo, cliente_edad, mpasivos_margen
#
# De estas, 4 corresponden a productos que NO todos los clientes
# tienen (alto riesgo de 0 / NA si se usan como denominador):
#   cpayroll_trx, mcuenta_corriente, mprestamos_personales,
#   Visa_mpagominimo
#
# Estrategia:
#   A) Para esas 4, se agrega un flag explicito "tiene o no el
#      producto" (informacion limpia, sin depender de como LightGBM
#      interprete el NA de un ratio).
#   B) Se generan los 10*9 = 90 ratios cruzados ordenados (A/B, A != B)
#      entre las 10 variables, usando divseg() (denominador 0 -> NA).
#      LightGBM maneja el NA nativamente (default direction por split),
#      pero el flag del punto A) le da al arbol una version limpia de
#      esa misma señal para los 4 denominadores mas problematicos.
#   C) Se imprime el % de NA/0 por variable generada, para poder
#      filtrar manualmente antes de sumarlas a campos_buenos si hiciera
#      falta.
# ---------------------------------------------------------------------

vars_top10 <- c("ctrx_quarter", "mcaja_ahorro", "cpayroll_trx", "mcuentas_saldo",
                 "mcuenta_corriente", "mprestamos_personales", "mtarjeta_visa_consumo",
                 "Visa_mpagominimo", "cliente_edad", "mpasivos_margen")

vars_alto_riesgo <- c("cpayroll_trx", "mcuenta_corriente",
                       "mprestamos_personales", "Visa_mpagominimo")

# --- A) flags explicitos "tiene o no el producto" para los denominadores riesgosos ---
if( atributos_presentes( vars_alto_riesgo ) )
{
  for( v in vars_alto_riesgo )
  {
    nombre_flag <- paste0( "flag_tiene_", v )
    dataset[, (nombre_flag) := as.integer( get(v) > 0 ) ]
  }
}

# --- B) 90 ratios cruzados ordenados entre las 10 variables ---
if( atributos_presentes( vars_top10 ) )
{
  for( num in vars_top10 )
  {
    for( den in vars_top10 )
    {
      if( num == den ) next   # A/A no tiene sentido, se descarta

      nombre_nueva <- paste0( "ratio_", num, "_sobre_", den )
      dataset[, (nombre_nueva) := divseg( get(num), get(den) ) ]
    }
  }
}



In [ ]:
# visualizo las columas del dataset a esta etapa
colnames(dataset)

[1] "numero_de_cliente"                                        
  [2] "foto_mes"                                                 
  [3] "internet"                                                 
  [4] "cliente_edad"                                             
  [5] "cliente_antiguedad"                                       
  [6] "mrentabilidad"                                            
  [7] "mrentabilidad_annual"                                     
  [8] "mcomisiones"                                              
  [9] "mactivos_margen"                                          
 [10] "mpasivos_margen"                                          
 [11] "cproductos"                                               
 [12] "mcuenta_corriente"                                        
 [13] "mcaja_ahorro"                                             
 [14] "cdescubierto_preacordado"                                 
 [15] "mcuentas_saldo"                                           
 [16] "ctarjeta_visa_transacciones"                              
 [17] "mtarjeta_visa_consumo"                                    
 [18] "mtarjeta_master_consumo"                                  
 [19] "mprestamos_personales"                                    
 [20] "cpayroll_trx"                                             
 [21] "mpayroll"                                                 
 [22] "ccomisiones_mantenimiento"                                
 [23] "ccallcenter_transacciones"                                
 [24] "chomebanking_transacciones"                               
 [25] "ctrx_quarter"                                             
 [26] "Master_status"                                            
 [27] "Master_fechaalta"                                         
 [28] "Master_mpagominimo"                                       
 [29] "Visa_status"                                              
 [30] "Visa_fechaalta"                                           
 [31] "Visa_mpagominimo"                                         
 [32] "clase_ternaria"                                           
 [33] "kmes"                                                     
 [34] "mpayroll_sobre_edad"                                      
 [35] "flag_tiene_cpayroll_trx"                                  
 [36] "flag_tiene_mcuenta_corriente"                             
 [37] "flag_tiene_mprestamos_personales"                         
 [38] "flag_tiene_Visa_mpagominimo"                              
 [39] "ratio_ctrx_quarter_sobre_mcaja_ahorro"                    
 [40] "ratio_ctrx_quarter_sobre_cpayroll_trx"                    
 [41] "ratio_ctrx_quarter_sobre_mcuentas_saldo"                  
 [42] "ratio_ctrx_quarter_sobre_mcuenta_corriente"               
 [43] "ratio_ctrx_quarter_sobre_mprestamos_personales"           
 [44] "ratio_ctrx_quarter_sobre_mtarjeta_visa_consumo"           
 [45] "ratio_ctrx_quarter_sobre_Visa_mpagominimo"                
 [46] "ratio_ctrx_quarter_sobre_cliente_edad"                    
 [47] "ratio_ctrx_quarter_sobre_mpasivos_margen"                 
 [48] "ratio_mcaja_ahorro_sobre_ctrx_quarter"                    
 [49] "ratio_mcaja_ahorro_sobre_cpayroll_trx"                    
 [50] "ratio_mcaja_ahorro_sobre_mcuentas_saldo"                  
 [51] "ratio_mcaja_ahorro_sobre_mcuenta_corriente"               
 [52] "ratio_mcaja_ahorro_sobre_mprestamos_personales"           
 [53] "ratio_mcaja_ahorro_sobre_mtarjeta_visa_consumo"           
 [54] "ratio_mcaja_ahorro_sobre_Visa_mpagominimo"                
 [55] "ratio_mcaja_ahorro_sobre_cliente_edad"                    
 [56] "ratio_mcaja_ahorro_sobre_mpasivos_margen"                 
 [57] "ratio_cpayroll_trx_sobre_ctrx_quarter"                    
 [58] "ratio_cpayroll_trx_sobre_mcaja_ahorro"                    
 [59] "ratio_cpayroll_trx_sobre_mcuentas_saldo"                  
 [60] "ratio_cpayroll_trx_sobre_mcuenta_corriente"               
 [61] "ratio_cpayroll_trx_sobre_mprestamos

#### 6.3.1.4  FE_rf Feature Engineering de nuevas variables a partir de hojas de Random Forest

Esto se mostrará unicamente a la *modalidad Analista Sr*

In [ ]:
# No se implementa Feature Engineering a partir de Random Forest

#### 6.3.1.5  FEhist Feature Engineering historico

El Feature Engineering Histórico es la etapa que más aporta a la ganancia final, ya que enriquece cada registro del dataset con su historia.

Para cada campo del dataset original (*)
se crean lo siguientes campos de a partir de la historia
* lag1  lags de orden 1
* delta1  =  valor actual - lag1
* lag2  lags de orden 2
* delta2  = valor actual - lag2


(*) Excepto para los campos  <numero_de_cliente,  foto_mes,  clase_ternaria>

In [ ]:
# Feature Engineering Historico

# todo es lagueable, menos la primary key y la clase
cols_lagueables <- copy( setdiff(
    colnames(dataset),
    c("numero_de_cliente", "foto_mes", "clase_ternaria")
) )

# https://rdrr.io/cran/data.table/man/shift.html

# lags de orden 1
dataset[,
    paste0(cols_lagueables, "_lag1") := shift(.SD, 1, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# lags de orden 2
dataset[,
    paste0(cols_lagueables, "_lag2") := shift(.SD, 2, NA, "lag"),
    by = numero_de_cliente,
    .SDcols = cols_lagueables
]

# agrego los delta lags
for (vcol in cols_lagueables)
{
    dataset[, paste0(vcol, "_delta1") := get(vcol) - get(paste0(vcol, "_lag1"))]
    dataset[, paste0(vcol, "_delta2") := get(vcol) - get(paste0(vcol, "_lag2"))]
}


Verificacion de los campos recien creados

In [ ]:
ncol(dataset)
colnames(dataset)

[1] 903

[1] "numero_de_cliente"                                               
  [2] "foto_mes"                                                        
  [3] "internet"                                                        
  [4] "cliente_edad"                                                    
  [5] "cliente_antiguedad"                                              
  [6] "mrentabilidad"                                                   
  [7] "mrentabilidad_annual"                                            
  [8] "mcomisiones"                                                     
  [9] "mactivos_margen"                                                 
 [10] "mpasivos_margen"                                                 
 [11] "cproductos"                                                      
 [12] "mcuenta_corriente"                                               
 [13] "mcaja_ahorro"                                                    
 [14] "cdescubierto_preacordado"                                        
 [15] "mcuentas_saldo"                                                  
 [16] "ctarjeta_visa_transacciones"                                     
 [17] "mtarjeta_visa_consumo"                                           
 [18] "mtarjeta_master_consumo"                                         
 [19] "mprestamos_personales"                                           
 [20] "cpayroll_trx"                                                    
 [21] "mpayroll"                                                        
 [22] "ccomisiones_mantenimiento"                                       
 [23] "ccallcenter_transacciones"                                       
 [24] "chomebanking_transacciones"                                      
 [25] "ctrx_quarter"                                                    
 [26] "Master_status"                                                   
 [27] "Master_fechaalta"                                                
 [28] "Master_mpagominimo"                                              
 [29] "Visa_status"                                                     
 [30] "Visa_fechaalta"                                                  
 [31] "Visa_mpagominimo"                                                
 [32] "clase_ternaria"                                                  
 [33] "kmes"                                                            
 [34] "mpayroll_sobre_edad"                                             
 [35] "flag_tiene_cpayroll_trx"                                         
 [36] "flag_tiene_mcuenta_corriente"                                    
 [37] "flag_tiene_mprestamos_personales"                                
 [38] "flag_tiene_Visa_mpagominimo"                                     
 [39] "ratio_ctrx_quarter_sobre_mcaja_ahorro"                           
 [40] "ratio_ctrx_quarter_sobre_cpayroll_trx"                           
 [41] "ratio_ctrx_quarter_sobre_mcuentas_saldo"                         
 [42] "ratio_ctrx_quarter_sobre_mcuenta_corriente"                      
 [43] "ratio_ctrx_quarter_sobre_mprestamos_personales"                  
 [44] "ratio_ctrx_quarter_sobre_mtarjeta_visa_consumo"                  
 [45] "ratio_ctrx_quarter_sobre_Visa_mpagominimo"                       
 [46] "ratio_ctrx_quarter_sobre_cliente_edad"                           
 [47] "ratio_ctrx_quarter_sobre_mpasivos_margen"                        
 [48] "ratio_mcaja_ahorro_sobre_ctrx_quarter"                           
 [49] "ratio_mcaja_ahorro_sobre_cpayroll_trx"                           
 [50] "ratio_mcaja_ahorro_sobre_mcuentas_saldo"                         
 [51] "ratio_mcaja_ahorro_sobre_mcuenta_corriente"                      
 [52] "ratio_mcaja_ahorro_sobre_mprestamos_personales"                  
 [53] "ratio_mcaja_ahorro_sobre_mtarjeta_visa_consumo"                  
 [54] "ratio_mcaja_ahorro_sobre_Visa_mpagominimo"                       
 [55] "ratio_mcaja_ahorro_sobre_cliente_edad"               

#### 6.3.1.6  FEhist Reduccion dimensionalidad con canaritos

Esta etapa solo se mostrará a la *modalidad Anlista Sr* por algun canal secreto de forma de no confundir a los *Analista Jr*  nni distraer con detalles operativos a la estratégica *Modalidad Gerencial*

In [ ]:
# No se implementa la reduccion de la dimensionalidad con canaritos

### 6.3.2 Modelado

#### 6.3.2.1 Training Strategy

Esta etapa de Workflow de  Training Strategy esta pensada para la *Modalidad Gerencial* que posee el dataset de [202005, 202109]
<br> Si usted es un Analista, posee el periodo de [201901, 202109] y deberá experimentar en que meses le conviene experimentar

<br> A la *Modalidad Gerencial* no se le complicada la vida con el undersampling de los continua, por eso PARAM$trainingstrategy$training_pct <- 1.0
<br> Sin embargo, si usted es  *Analista SR* posee un dataset 50 veces ( filas x columnas) más grande que la *Modalidad Gerencial*  y por un tema de velocidad y experimentación más rápida puede llegar a necesitar activar el undersampling de la clase mayoritaria, a pesar de estar corriendo en Google Cloud.

Se hace una estrategia de entrenamiento muy sencilla, tomando todos los meses posibles, SIN eliminar nada x pandemia ni por ningun otro motivo

* future = 202107  obviamente completo

* final_train =  [ 202005, 202105 ]  SIN undersampling

* training
   * testing = NO HAY
   * validation =  202105   completo, sin undersampling
   * training = [ 202005, 202104 ]  donde se consideran el 100% de los CONTINUA

In [ ]:
PARAM$trainingstrategy$validate <- c(202105)

PARAM$trainingstrategy$training <- c(
  202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007,
  202006, 202005
)

PARAM$trainingstrategy$training_pct <- 1.0


PARAM$trainingstrategy$positivos <- c( "BAJA+1", "BAJA+2")

In [ ]:
# seteo la clase01   1={BAJA+1, BAJA+2}   0={CONTINUA}
dataset[, clase01 := ifelse( clase_ternaria %in% PARAM$trainingstrategy$positivos, 1, 0 )]

In [ ]:
# los campos en los que se entrena
campos_buenos <- copy( setdiff(
    colnames(dataset), c("clase_ternaria","clase01","azar"))
)

La siguiente celda corre en interminables 8 minutos

In [ ]:
# preparo para que se pueda hacer undersampling de los CONTINUA
#  solamente por un tema de VELOCIDAD
set.seed(PARAM$semilla_primigenia, kind = "L'Ecuyer-CMRG")
dataset[, azar:=runif(nrow(dataset))]

# undersampling de los CONTINUA
dataset[, fold_train :=  foto_mes %in%  PARAM$trainingstrategy$training &
    (clase_ternaria %in% c("BAJA+1", "BAJA+2") |
     azar < PARAM$trainingstrategy$training_pct ) ]


if( !require("lightgbm")) install.packages("lightgbm")
require("lightgbm")

dtrain <- lgb.Dataset(
  data= data.matrix(dataset[fold_train == TRUE, campos_buenos, with = FALSE]),
  label= dataset[fold_train == TRUE, clase01],
  free_raw_data= TRUE
)

Loading required package: lightgbm

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘lightgbm’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Loading required package: lightgbm



In [ ]:
# datos de validation
dvalidate <- lgb.Dataset(
  data= data.matrix(dataset[foto_mes %in% PARAM$trainingstrategy$validate, campos_buenos, with = FALSE]),
  label= dataset[foto_mes %in% PARAM$trainingstrategy$validate, clase01],
  free_raw_data= TRUE
)

nrow(dvalidate)

[1] 13167

####  6.3.2.2. Hyperparameter Tuning

* Clase binaria que se optimiza :  positivos = [ BAJA+1, BAJA+2 ]

* Metrica que se optimiza **AUC** Area Under Curve de la  ROC Curve

es muy importante notar que intencionalmente  **NO** se está optimizando la funcion de ganancia del problema

* Parametros no default, fijos de LightGBM que no se optimizan
  * max_bin = 31 , Alienigenas Ancestrales contruyeron las pirámides y dejaron a la humanidad en un jeroglifico  *max_bin=31*
  * feature_fraction = 0.5  para poner algo que generalmente no falla
  * learning_rate = 0.03  para que aprenda lento


* Parametros que se optimizan en el Grid Search
  * num_leaves  [64, 512]
  * min_data_in_leaf  [64, 2048]

In [ ]:
# parametros fijos del LightGBM
PARAM$lgbm$param_fijos <- list(
  objective= "binary",
  metric= "auc",
  first_metric_only= TRUE,
  boost_from_average= TRUE,
  feature_pre_filter= FALSE,
  verbosity= -100,
  force_row_wise= TRUE, # para evitar warning
  seed= PARAM$semilla_primigenia,
  max_bin= 31,
  learning_rate= 0.03,
  feature_fraction= 0.5,
  num_iterations= 2048,  # valor grande, lo limita early_stopping_rounds
  early_stopping_rounds= 200,
  num_leaves= 64,
  min_data_in_leaf= 128
)


In [ ]:
# En  x llegan los parametros moviles de LightGBM
#  devuelve la AUC en validate del modelo entrenado
#  en el parametro x llegan los hiperparámetros que se estan optimizando

Estimar_AUC_lightgbm <- function(x) {

  # x pisa (o agrega) a param_fijos
  param_completo <- modifyList(PARAM$lgbm$param_fijos, x)

  # entreno LightGBM
  modelo_train <- lgb.train(
    data= dtrain,
    valids= list(valid = dvalidate),
    eval= "auc",
    param= param_completo,
    verbose= -100
  )

  # recupero la AUC en validation
  AUC <- modelo_train$record_evals$valid$auc$eval[[modelo_train$best_iter]]

  message(format(Sys.time(), "%a %b %d %X %Y  "),
    toString(x),
    " niter ", modelo_train$best_iter,
    " AUC ", AUC
  )

  niter <- modelo_train$best_iter
  # hago espacio en la memoria
  rm(modelo_train)
  gc(full= TRUE, verbose= FALSE)

  return( list(AUC, niter))
}

seteo del Grid Search

In [ ]:
# lo que sigue a continuacion es una forma alternativa a los loops anidados
# creo una tabla con el producto cartesiano de los vectores
tb_nueva <- CJ(
  num_leaves= c(64, 128, 256, 512),
  min_data_in_leaf= c(64, 256, 512, 1024, 2048)
)

Corrida del Grid Search,  aqui se hace el trabajo pesado
<br> por favor no se asuste con los warnings que pudieran aparecer
<br> ATENCION, la siguiente celda demora 60 minutos en Colab
<br> lamento profundamente tal intolerable espera gerencial

In [ ]:
# registro a registro calculo la AUC
tb_nueva[, c("AUC", "num_iterations"):= Estimar_AUC_lightgbm( .SD ),
  by=1:nrow(tb_nueva) ]

Mon Sep 14 12:53:34 PM 2026  64, 64 niter 696 AUC 0.943710804821916

Mon Sep 14 12:57:40 PM 2026  64, 256 niter 389 AUC 0.94632581299248

Mon Sep 14 01:03:06 PM 2026  64, 512 niter 570 AUC 0.948606298606299

Mon Sep 14 01:10:21 PM 2026  64, 1024 niter 772 AUC 0.950165122387345

Mon Sep 14 01:20:24 PM 2026  64, 2048 niter 1138 AUC 0.949728360839472

Mon Sep 14 01:24:42 PM 2026  128, 64 niter 326 AUC 0.947846992291437

Mon Sep 14 01:28:56 PM 2026  128, 256 niter 304 AUC 0.949845572067794

Mon Sep 14 01:34:41 PM 2026  128, 512 niter 469 AUC 0.948206325984104

Mon Sep 14 01:44:12 PM 2026  128, 1024 niter 913 AUC 0.946947802503358

Mon Sep 14 01:55:09 PM 2026  128, 2048 niter 1253 AUC 0.949213743658188

Mon Sep 14 02:00:53 PM 2026  256, 64 niter 368 AUC 0.948550259661371

Mon Sep 14 02:07:22 PM 2026  256, 256 niter 427 AUC 0.949414371636594

Mon Sep 14 02:15:22 PM 2026  256, 512 niter 592 AUC 0.950029088917978

Mon Sep 14 02:21:27 PM 2026  256, 1024 niter 490 AUC 0.947291308402419

Mon Sep 

la optimizacion de hiperparámetros de tipo  Grid Search ha corrido, extraigo los mejores hiperparametros

In [ ]:
tb_nueva

fwrite( tb_nueva,
  file= "tb_grid_search_01.txt",
  sep="\t",
  append= TRUE
)

num_leaves,min_data_in_leaf,AUC,num_iterations
<dbl>,<dbl>,<dbl>,<int>
64,64,0.9437108,696
64,256,0.9463258,389
64,512,0.9486063,570
64,1024,0.9501651,772
64,2048,0.9497284,1138
128,64,0.9478470,326
128,256,0.9498456,304
128,512,0.9482063,469
128,1024,0.9469478,913


In [ ]:
setorder( tb_nueva, -AUC)  # ordeno DESCENDENTE por AUC
PARAM$out$lgbm$AUC <- tb_nueva[1, AUC] # en la posicion 1 estan los mejores
PARAM$out$lgbm$mejores_hiperparametros <- as.list( tb_nueva[1] )
PARAM$out$lgbm$mejores_hiperparametros$AUC <- NULL
PARAM$out$lgbm$mejores_hiperparametros

$num_leaves
[1] 64

$min_data_in_leaf
[1] 1024

$num_iterations
[1] 772

### 6.3.3 Produccion

#### Final Training
Construyo el modelo final, que es uno solo, no hace ningun tipo de particion < training, validation, testing>]

##### Final Training Dataset

Aqui esta la gran decision de en qué meses hago el Final Training
<br> debo utilizar los mejores hiperparámetros que encontré en la optimización de hiperparámetros

In [ ]:
PARAM$trainingstrategy$final_train <- c(
  202105, 202104, 202103, 202102, 202101,
  202012, 202011, 202010, 202009, 202008, 202007,
  202006, 202005
)

dataset[, fold_final_train := foto_mes %in% PARAM$trainingstrategy$final_train ]

# creo el dfinal_train en formato  LightGBM
dfinal_train <- lgb.Dataset(
  data= data.matrix(dataset[fold_final_train == TRUE, campos_buenos, with= FALSE]),
  label= dataset[fold_final_train == TRUE, clase01],
  free_raw_data= TRUE
)

nrow( dfinal_train) # verifico el tamaño

[1] 166250

##### Final Training Hyperparameters

In [ ]:
# uno los parametros fijos y los mejores encontrados de los variables
fijos <- copy(PARAM$lgbm$param_fijos)

# quito lo que optimice en la Bayesian Optimization
fijos$num_iterations <- NULL
fijos$early_stopping_rounds <- NULL

# agrego a los hiperparametros fijos los que encontre con la Bayesian Optimization
param_final <- c(fijos, PARAM$out$lgbm$mejores_hiperparametros)

##### Training
Genero el modelo final, siempre sobre TODOS los datos de  final_train, sin hacer ningun tipo de undersampling de la clase mayoritaria

#### Scoring

Defino el mes de futuro sobre el que voy a evaluar cada modelo (uno por semilla).

In [ ]:
PARAM$trainingstrategy$future <- c(202107)

dfuture <- dataset[ foto_mes %in% PARAM$trainingstrategy$future ]


##### Entrenamiento y evaluacion sobre MULTIPLES SEMILLAS

Pauta general del Experimento Colaborativo: se necesitan como minimo 5 semillas
propias del grupo (no mas de 15) para poder comparar variantes (p.ej. baseline
vs. FE nuevo) con el Test de Wilcoxon. Aca se entrena un modelo final por cada
semilla de `PARAM$semillas`, manteniendo Ceteris Paribus los mismos
hiperparametros (fijos + los mejores de la Bayesian Optimization, que SI se
corre una unica vez, no por semilla, ver 6.3.2.2).

In [ ]:
tb_semillas <- data.table()      # 1 fila por semilla: ganancia maxima y envios optimo
lista_curvas <- list()           # curva completa (envios vs gan_suavizada) de cada semilla
lista_prob   <- list()           # probabilidad predicha por cada semilla, para promediar
lista_import <- list()           # importancia de variables de cada semilla, para promediar

for( s in PARAM$semillas )
{
  param_final_s <- copy(param_final)
  param_final_s$seed <- s

  set.seed(s, kind = "L'Ecuyer-CMRG")

  modelo_s <- lgb.train(
    data= dfinal_train,
    param= param_final_s,
    verbose= -100
  )

  prediccion_s <- predict(
    modelo_s,
    data.matrix(dfuture[, campos_buenos, with= FALSE])
  )

  lista_prob[[ as.character(s) ]] <- prediccion_s
  lista_import[[ as.character(s) ]] <- lgb.importance( modelo_s )

  tb_pred_s <- dfuture[, list(numero_de_cliente, clase_ternaria)]
  tb_pred_s[, prob := prediccion_s]

  # asigno ganancias
  tb_pred_s[, ganancia := -0.025 ]
  tb_pred_s[ clase_ternaria== "BAJA+2", ganancia := 0.975 ]

  # reordeno, y acumulada
  setorder( tb_pred_s, -prob )
  tb_pred_s[, gan_acum := cumsum(ganancia) ]

  # media movil de ancho 400
  tb_pred_s[, gan_suavizada := frollmean(
      x= gan_acum, n= 400, align= "center", na.rm= TRUE, hasNA= TRUE) ]
  tb_pred_s[, envios := .I ]

  lista_curvas[[ as.character(s) ]] <- tb_pred_s[, list(envios, gan_suavizada)]

  tb_semillas <- rbind( tb_semillas,
    data.table(
      semilla= s,
      ganancia_suavizada_max= max( tb_pred_s$gan_suavizada, na.rm= TRUE ),
      envios_optimo= which.max( tb_pred_s$gan_suavizada )
    )
  )
}

tb_semillas   # ganancia obtenida en CADA semilla


<0 x 0 matrix>

##### Tabla Prediccion

In [ ]:
# probabilidad promedio entre las N semillas (ensemble simple)
#  me va a ser util para hacer Ensembles de modelos con otros grupos/etapas
tb_prediccion <- dfuture[, list(numero_de_cliente)]
tb_prediccion[, prob := Reduce(`+`, lista_prob) / length(lista_prob) ]

fwrite(tb_prediccion,
  file= "prediccion.txt",
  sep= "\t"
)


##### Importancia de Variables (promedio entre semillas)

In [ ]:
# promedio la ganancia (Gain) de cada variable entre las N semillas
#  sirve para verificar objetivamente si las variables nuevas de FE aportan
tb_importancia <- rbindlist( lista_import, idcol= "semilla" )
tb_importancia_prom <- tb_importancia[, list(Gain= mean(Gain)), by= Feature]
setorder( tb_importancia_prom, -Gain )

# marco cuales son las variables nuevas creadas en este problema
variables_fe_nuevas <- c("actividad_total","ratio_uso_tc_total",
                          "ratio_consumo_limite_visa","saldo_financiado_visa")
tb_importancia_prom[, es_variable_nueva := Feature %in% variables_fe_nuevas]

fwrite( tb_importancia_prom, file= "importancia_variables.txt", sep= "\t" )

# top 20 general, y el ranking puntual de las 4 variables nuevas
head( tb_importancia_prom, 20 )
tb_importancia_prom[ es_variable_nueva == TRUE ]


ERROR: Error in eval(bysub, parent.frame(), parent.frame()): object 'Feature' not found


#### Curva de Ganancia

Genero las salidas: reporto el **promedio y desvio estandar** de la ganancia sobre las N semillas (no un unico numero), tal como exige la naturaleza probabilistica del problema.

In [ ]:
# Este es el resultado a reportar: PROMEDIO y DESVIO ESTANDAR sobre las N semillas
resultado <- list()
resultado$semillas                    <- PARAM$semillas
resultado$ganancia_por_semilla        <- tb_semillas$ganancia_suavizada_max
resultado$ganancia_suavizada_max      <- mean( tb_semillas$ganancia_suavizada_max )   # promedio
resultado$ganancia_suavizada_sd       <- sd( tb_semillas$ganancia_suavizada_max )
resultado$envios                      <- round( mean( tb_semillas$envios_optimo ) )

options(digits= 8)
resultado


In [ ]:
# grabo las ganancias de cada semilla, y la curva promedio
fwrite( tb_semillas, file= "ganancias_por_semilla.txt", sep= "\t" )

tb_todas    <- rbindlist( lista_curvas, idcol= "semilla" )
tb_promedio <- tb_todas[, list(gan_suavizada= mean(gan_suavizada, na.rm= TRUE)), by= envios]

fwrite( tb_promedio, file= "ganancias.txt", sep= "\t" )


In [ ]:
# genero el grafico: curvas individuales (gris) + curva promedio (azul)
pdf("curva_de_ganancia.pdf")

plot(
  x= tb_promedio$envios,
  y= tb_promedio$gan_suavizada,
  type= "n",
  xlim= c(0, 6000),
  ylim= c(0, 50),
  main= paste0(
    "Gan prom= ", as.integer(resultado$ganancia_suavizada_max),
    "  (sd= ", round(resultado$ganancia_suavizada_sd, 1), ")",
    "  envios prom= ", resultado$envios,
    "  (n=", length(PARAM$semillas), " semillas)"
  ),
  xlab= "Envios",
  ylab= "Ganancia",
  panel.first= grid()
)

for( s in PARAM$semillas ) {
  curva <- lista_curvas[[ as.character(s) ]]
  lines( curva$envios, curva$gan_suavizada, col= "gray80" )
}

lines( tb_promedio$envios, tb_promedio$gan_suavizada, col= "blue", lwd= 2 )

dev.off()


In [ ]:
# grabo los parametros
if( !require("yaml")) install.packages("yaml")
require("yaml")

PARAM$resultado <- resultado

write_yaml( PARAM, file="PARAM.yml")


In [ ]:
format(Sys.time(), "%a %b %d %X %Y")